# Quikr Car Data — Cleaning Notebook

This notebook takes the raw, messy data scraped from Quikr (`quikr_car.csv`) and cleans it into a model-ready file (`Cleaned_Car_data.csv`).

**Why this step matters:** real-world scraped data is almost never usable as-is. Before any model can be trained, the data has to be inspected for inconsistent types, missing values, and junk entries. This notebook walks through that process step by step, explaining *why* each fix is needed, not just *what* code does it.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the Raw Data

In [ ]:
car = pd.read_csv('quikr_car.csv')
car.head()

In [ ]:
car.shape

In [ ]:
car.info()

## 3. Create a Backup Copy

Good practice before destructive cleaning: keep an untouched copy of the raw data in case a cleaning step needs to be revisited.

In [ ]:
backup = car.copy()

## 4. Data Quality Issues

Inspecting the raw data reveals several problems:

- **`name`** values are inconsistent — some include spammy text like *'Maruti Ertiga showroom condition with'*.
- **`company`** sometimes isn't a real company at all — values like `'Used'` or `'URJENT'` slipped in from bad scraping.
- **`year`** contains many non-year values (it's stored as text/object, not a number).
- **`Price`** contains the literal text `'Ask For Price'` instead of a number, and numeric prices include commas (e.g. `'80,000'`), so it can't be read as an integer yet.
- **`kms_driven`** has the unit `'kms'` attached to every value (e.g. `'45,000 kms'`), commas inside the numbers, some missing values, and a couple of rows where `'Petrol'` ended up in this column due to a scraping misalignment.
- **`fuel_type`** has missing (NaN) values.

**The key takeaway here is:** in a real dataset, almost every column can have a different kind of problem — type mismatches, embedded units, missing values, and outright garbage values. Cleaning isn't one fix; it's a fix per column.

## 5. Cleaning `year`

`year` is stored as text and contains non-year junk. Keep only rows where the value is purely numeric, then convert the column to an integer type.

In [ ]:
car = car[car['year'].str.isnumeric()]

In [ ]:
car['year'] = car['year'].astype(int)

## 6. Cleaning `Price`

Two problems: some rows literally say `'Ask For Price'` instead of a number (these have no usable price, so they're dropped), and the numeric values contain commas that block conversion to `int`.

In [ ]:
car = car[car['Price'] != 'Ask For Price']

In [ ]:
car['Price'] = car['Price'].str.replace(',', '').astype(int)

## 7. Cleaning `kms_driven`

Each value has the unit `'kms'` stuck to it (e.g. `'45,000 kms'`). Splitting on whitespace and taking the first chunk isolates the number; removing commas makes it convertible to `int`.

In [ ]:
car['kms_driven'] = car['kms_driven'].str.split().str.get(0).str.replace(',', '')

After that split, a few rows are left with non-numeric junk — some are missing values, and a couple actually contain `'Petrol'` (a scraping error that shifted columns). Filtering to only numeric values removes both problems in one step.

In [ ]:
car = car[car['kms_driven'].str.isnumeric()]

In [ ]:
car['kms_driven'] = car['kms_driven'].astype(int)

## 8. Cleaning `fuel_type`

Drop rows where fuel type is missing — there's no reliable way to guess it, and it's a required model feature.

In [ ]:
car = car[~car['fuel_type'].isna()]

In [ ]:
car.shape

## 9. Cleaning `name`

Note: the spammy `name`/`company` values mentioned in Section 4 have *already* been removed as a side effect of the filters above (they mostly belonged to the same bad rows dropped from `year`, `Price`, and `kms_driven`).

What's left is trimmed to just the first three words (e.g. `'Maruti Suzuki Swift Dzire VDI'` becomes `'Maruti Suzuki Swift'`) — this keeps the car model readable without an explosion of near-duplicate, overly specific trim-level names.

In [ ]:
car['name'] = car['name'].str.split().str.slice(start=0, stop=3).str.join(' ')

## 10. Reset the Index

After all the row filtering above, the index has gaps (e.g. 0, 1, 4, 7, ...). Resetting it gives a clean, continuous index for the final dataset.

In [ ]:
car = car.reset_index(drop=True)

## 11. Cleaned Data — Final Check

In [ ]:
car

In [ ]:
car.info()

In [ ]:
car.describe(include='all')

## 12. Save the Cleaned Dataset

In [ ]:
car.to_csv('Cleaned_Car_data.csv')

---
### Recap

- Started with **892** raw scraped rows across 6 columns, all stored as text.
- Ended with **816** clean rows with correct types (`year`, `Price`, `kms_driven` as integers) and no missing values.
- Every row dropped was dropped for a specific, inspectable reason — never silently.

**In interviews, this exact topic is usually tested by asking:** *"Walk me through how you'd clean a messy real-world dataset"* or *"How do you decide whether to drop or impute a missing value?"* Being able to explain the specific reasoning per column (as this notebook does) is exactly what that question is checking for.

**Portfolio idea:** take one more messy scraped dataset (e.g. from Kaggle's "messy data" datasets) and write a similar cleaning notebook, but add automated checks — e.g. `assert car.isna().sum().sum() == 0` at the end — to show you think about validating cleanliness, not just performing it.